# 05 — Evaluation and Experimentation

All numbers are loaded from JSON reports; no computation happens in this notebook.

In [1]:
import json
from pathlib import Path

EVAL = Path("../reports/evaluation")
RANKER = Path("../reports/ranker")

def load(name):
    return json.loads((EVAL / name).read_text())

rolling  = load("rolling_origin.json")
suite    = load("metric_suite.json")
segs     = load("segment_analysis.json")
ab       = load("ab_test.json")
protocol = load("holdout_protocol.json")
holdout  = load("holdout_results.json")


## A — Robustness Across Weeks (Rolling Origin)

Three expanding-window models trained on increasing numbers of folds, evaluated on the next held-out fold.

In [2]:
print(f"Mean lift vs heuristic: {rolling['mean_lift_vs_heuristic']:+.6f}")
print(f"Lift holds every week (all CIs exclude 0): {rolling['lift_holds_every_week']}")
print()
print(f"{'Fold':>6s}  {'Train folds':>15s}  {'Ranker MAP@12':>14s}  {'Heuristic':>12s}  "
      f"{'Lift CI lo':>12s}  {'Lift CI hi':>12s}  {'Note'}")
for r in rolling['folds']:
    boot = r['boot_vs_heuristic']
    folds_str = str(r['train_folds'])
    print(f"  {r['fold']:>4d}  {folds_str:>15s}  {r['ranker_map@12']:>14.6f}  "
          f"{r['heuristic_map@12']:>12.6f}  {boot['ci_lo']:>+12.6f}  {boot['ci_hi']:>+12.6f}  "
          f"{r.get('note','')[:40]}")


Mean lift vs heuristic: +0.011303
Lift holds every week (all CIs exclude 0): True

  Fold      Train folds   Ranker MAP@12     Heuristic    Lift CI lo    Lift CI hi  Note
   101            [100]        0.029125      0.019198     +0.009414     +0.010500  unbiased
   102       [100, 101]        0.035270      0.022445     +0.012208     +0.013401  optimistic: fold 102 was the tuning fold
   103  [100, 101, 102]        0.035832      0.024675     +0.010563     +0.011758  final model; eval fold is cleanly held o


**Interpretation.** Lift holds on every validation week.
- Fold 102 is marked OPTIMISTIC because the tuning hyperparameters were selected using fold-102 performance.
- Fold 103 is the uncontaminated estimate; it gives the CI used throughout this report.


## B — Metric Suite (fold 103)

In [3]:
r_acc = suite['ranker']['accuracy']
h_acc = suite['heuristic']['accuracy']
r_ba  = suite['ranker']['beyond_accuracy']
h_ba  = suite['heuristic']['beyond_accuracy']

rows = [
    ("MAP@12",              r_acc['map@12'],                        h_acc['map@12']),
    ("MAP@12 CI lo",        r_acc['map@12_ci']['ci_lo'],            h_acc['map@12_ci']['ci_lo']),
    ("MAP@12 CI hi",        r_acc['map@12_ci']['ci_hi'],            h_acc['map@12_ci']['ci_hi']),
    ("Hit Rate@12",         r_acc['hit_rate@12'],                   h_acc['hit_rate@12']),
    ("Hit Rate@12 CI lo",   r_acc['hit_rate@12_ci']['ci_lo'],       h_acc['hit_rate@12_ci']['ci_lo']),
    ("Hit Rate@12 CI hi",   r_acc['hit_rate@12_ci']['ci_hi'],       h_acc['hit_rate@12_ci']['ci_hi']),
    ("NDCG@12",             r_acc['ndcg@12'],                       "—"),
    ("Recall@12",           r_acc['recall@12'],                     "—"),
    ("Catalog Coverage",    r_ba['catalog_coverage'],               h_ba['catalog_coverage']),
    ("Pop Bias (mean rank)",r_ba['popularity_bias_mean_rank'],      h_ba['popularity_bias_mean_rank']),
    ("Repeat Share",        r_ba['repeat_share'],                   h_ba['repeat_share']),
    ("Novelty Share",       r_ba['novelty_share'],                  h_ba['novelty_share']),
]
print(f"{'Metric':<30s}  {'Ranker':>12s}  {'Heuristic':>12s}")
print("-" * 58)
for name, rv, hv in rows:
    r_str = f"{rv:.4f}" if isinstance(rv, float) else str(rv)
    h_str = f"{hv:.4f}" if isinstance(hv, float) else str(hv)
    print(f"  {name:<28s}  {r_str:>12s}  {h_str:>12s}")


Metric                                Ranker     Heuristic
----------------------------------------------------------
  MAP@12                              0.0358        0.0247
  MAP@12 CI lo                        0.0348        0.0238
  MAP@12 CI hi                        0.0368        0.0255
  Hit Rate@12                         0.1471        0.0857
  Hit Rate@12 CI lo                   0.1448        0.0838
  Hit Rate@12 CI hi                   0.1498        0.0880
  NDCG@12                             0.0529             —
  Recall@12                           0.0741             —
  Catalog Coverage                    0.1373        0.5046
  Pop Bias (mean rank)             9331.2481    12062.0479
  Repeat Share                        0.2391        0.8350
  Novelty Share                       0.0003        0.0003


**Trade-off interpretation.**
The ranker improves MAP@12 by +45% via popular personalised items (popularity bias: rank ~9,331 vs 12,062).
Lower repeat share (24% vs 83%): the ranker recommends items outside the customer's purchase history more often than the heuristic does.
The cost is narrower catalog coverage (14% vs 50%): the ranker concentrates on fewer high-signal articles.
Novelty share (items never sold before the cutoff) is near zero for both models — new-article cold-start is handled at the candidate stage, not the ranking stage.

> **Repeat share** = share of recommended items the customer bought before the cutoff.
> **Novelty share** = share of recommended items that had zero sales before the cutoff (a distinct, separate metric).


## C — Segment Analysis (fold 103, Holm-corrected)

All 21 segment tests; Holm–Bonferroni correction applied.  Ranker beats heuristic in every segment.

In [4]:
print(f"Segments tested: {segs['n_segments_tested']}  Flagged (CI ∩ 0): {segs['n_flagged']}")
print()
print(f"{'Family':<22s} {'Segment':<20s} {'n':>7s} {'Ranker':>9s} {'Heuristic':>10s} "
      f"{'p_adj':>8s} {'CI lo':>8s} {'CI hi':>8s}")
print("-" * 100)
for r in segs['results']:
    p_adj = r.get('p_adj', float('nan'))
    p_str = f"< 0.001" if p_adj < 0.001 else f"{p_adj:.4f}"
    print(f"  {r['family']:<20s} {r['segment']:<20s} {r['n']:>7,d} "
          f"{r['map@12_ranker']:>9.4f} {r['map@12_heuristic']:>10.4f} "
          f"{p_str:>8s} {r['ci_lo']:>+8.4f} {r['ci_hi']:>+8.4f}")


Segments tested: 21  Flagged (CI ∩ 0): 0

Family                 Segment                    n    Ranker  Heuristic    p_adj    CI lo    CI hi
----------------------------------------------------------------------------------------------------
  prior_purchases      0                      5,395    0.0097     0.0064   0.0420  +0.0023  +0.0044
  prior_purchases      1-4                    4,271    0.0433     0.0349   0.0420  +0.0059  +0.0107
  prior_purchases      5-19                  14,465    0.0388     0.0249   0.0420  +0.0126  +0.0154
  prior_purchases      20+                   47,888    0.0372     0.0258   0.0420  +0.0106  +0.0123
  age_bucket           <25                   19,525    0.0331     0.0206   0.0420  +0.0113  +0.0139
  age_bucket           25-34                 22,467    0.0342     0.0248   0.0420  +0.0084  +0.0104
  age_bucket           35-44                  7,623    0.0367     0.0272   0.0420  +0.0075  +0.0112
  age_bucket           45-54                 13,107    0.

In [5]:
print("Segment p-values: paired t-test on per-customer AP differences (Holm-adjusted)")
print(f"{'Family':<22s} {'Segment':<20s} {'n':>7s} {'p_ttest_adj':>12s} {'sig?':>5s}")
print("-" * 72)
for r in segs['results']:
    p_tt = r.get('p_ttest_adj', float('nan'))
    sig = r.get('significant_ttest_adj', False)
    p_str = f"{p_tt:.4e}" if p_tt == p_tt else "—"
    print(f"  {r['family']:<20s} {r['segment']:<20s} {r['n']:>7,d} {p_str:>12s} {'yes' if sig else 'no':>5s}")


Segment p-values: paired t-test on per-customer AP differences (Holm-adjusted)
Family                 Segment                    n  p_ttest_adj  sig?
------------------------------------------------------------------------
  prior_purchases      0                      5,395   4.8677e-09   yes
  prior_purchases      1-4                    4,271   2.8223e-11   yes
  prior_purchases      5-19                  14,465   2.0940e-74   yes
  prior_purchases      20+                   47,888  1.3398e-163   yes
  age_bucket           <25                   19,525   1.4359e-84   yes
  age_bucket           25-34                 22,467   1.7279e-64   yes
  age_bucket           35-44                  7,623   9.0820e-22   yes
  age_bucket           45-54                 13,107   1.1980e-44   yes
  age_bucket           55+                    8,976   3.1524e-41   yes
  age_bucket           missing                  321   2.2229e-02   yes
  channel_tertile      low_online            22,390   2.0751e-86   

**Notable segments.**
- *Recency ≥91 days*: +204% lift — the ranker revives inactive customers by surfacing relevant new-to-them items.
- *Buys new articles*: +72% lift — the ranker handles new-to-the-customer item discovery better than the heuristic.
- *Cold-start (0 prior purchases)*: +52% lift — even without history, the ranker's popularity and article features help.
- Bootstrap p-values (p_adj): resolution-limited at 2/(B+1) ≈ 0.002 for B=1000, so Holm-adjusted values are all ≈ 0.042. Paired t-test p-values (p_ttest_adj) give continuous resolution.


## D — Simulated A/B Test

**Assumption:** Offline replay assumes customers purchase the same items regardless of which list was shown (no feedback effect); online lift may differ.

### D.1 Power Analysis

In [6]:
pa = ab['power_analysis']
print(f"Control (heuristic) hit rate@12: {pa['p_control']:.4f}")
print(f"Treatment (ranker) hit rate@12:  {pa['p_treat']:.4f}")
print(f"Absolute lift:                   {pa['p_treat']-pa['p_control']:.4f}")
print()
print(f"Required n/arm (formula):     {pa['n_per_arm_formula']:,}")
print(f"Required n/arm (statsmodels): {pa['n_per_arm_statsmodels']:,}")
print(f"Cohen h:                       {pa['effect_size_h']:.4f}")
print(f"Weekly buying customers:       {pa['weekly_buying_customers_fold103']:,}")
print(f"Weekly per arm (50/50):        {pa['weekly_customers_per_arm']:,.0f}")
print(f"Weeks needed:                  {pa['weeks_needed']:.2f}")


Control (heuristic) hit rate@12: 0.0857
Treatment (ranker) hit rate@12:  0.1471
Absolute lift:                   0.0614

Required n/arm (formula):     427
Required n/arm (statsmodels): 421
Cohen h:                       0.1931
Weekly buying customers:       72,019
Weekly per arm (50/50):        36,010
Weeks needed:                  0.01


### D.2 MDE vs Sample-Size Curve

In [7]:
print(f"{'n/arm':>10s}  {'MDE (abs)':>12s}  {'MDE (rel %)':>12s}")
print("-" * 38)
p_ctrl = ab['power_analysis']['p_control']
for pt in ab['power_analysis']['mde_curve']:
    mde = pt['mde_absolute']
    mde_rel = 100*mde/p_ctrl
    print(f"  {pt['n_per_arm']:>8,d}  {mde:>12.4f}  {mde_rel:>11.1f}%")


     n/arm     MDE (abs)   MDE (rel %)
--------------------------------------
       500        0.0496         57.9%
     1,000        0.0351         40.9%
     2,000        0.0248         28.9%
     5,000        0.0157         18.3%
    10,000        0.0111         12.9%
    20,000        0.0078          9.2%
    36,000        0.0058          6.8%
    72,000        0.0041          4.8%


### D.3 Simulation Results (one split, seed=42)

In [8]:
sim = ab['simulation']
z = sim['hit_rate_z_test']
t = sim['ap_welch_t_test']
print("Hit rate@12 two-proportion z-test:")
print(f"  Arm A (heuristic): {z['p_a']:.4f}  n={z['n_a']:,}")
print(f"  Arm B (ranker):    {z['p_b']:.4f}  n={z['n_b']:,}")
print(f"  Effect:  {z['effect']:+.4f}  z={z['z']:.3f}  p={z['p_value']:.4f}")
print(f"  95% CI: [{z['ci_lo']:+.4f}, {z['ci_hi']:+.4f}]")
print()
print("AP@12 Welch t-test:")
print(f"  Arm A mean: {t['mean_a']:.6f}   Arm B mean: {t['mean_b']:.6f}")
print(f"  Effect:  {t['effect']:+.6f}  t={t['t']:.3f}  p={t['p_value']:.4f}")
print(f"  95% CI: [{t['ci_lo']:+.6f}, {t['ci_hi']:+.6f}]")
print()
print("Guardrails:")
ga = sim['guardrail_arm_a']
gb = sim['guardrail_arm_b']
print(f"  Arm A: coverage={ga['catalog_coverage']:.4f}  pop_rank={ga['popularity_bias_mean_rank']:.0f}")
print(f"  Arm B: coverage={gb['catalog_coverage']:.4f}  pop_rank={gb['popularity_bias_mean_rank']:.0f}")


Hit rate@12 two-proportion z-test:
  Arm A (heuristic): 0.0867  n=36,009
  Arm B (ranker):    0.1457  n=36,010
  Effect:  +0.0589  z=24.674  p=0.0000
  95% CI: [+0.0543, +0.0636]

AP@12 Welch t-test:
  Arm A mean: 0.024620   Arm B mean: 0.035201
  Effect:  +0.010581  t=11.006  p=0.0000
  95% CI: [+0.008697, +0.012466]

Guardrails:
  Arm A: coverage=0.4156  pop_rank=12033
  Arm B: coverage=0.1111  pop_rank=9333


### D.4 A/A Test (10,000 splits)

Both arms receive heuristic predictions. 95% Clopper–Pearson binomial CI on the observed FPR; calibrated if the nominal α=5% falls within the CI.

In [9]:
aa = ab['aa_test']
fpr = aa['false_positive_rate']
ci_lo = aa.get('fpr_ci_lo', float('nan'))
ci_hi = aa.get('fpr_ci_hi', float('nan'))
in_ci = aa.get('alpha_in_ci', aa.get('within_expected', '?'))
print(f"Both arms heuristic, {aa['n_sims']:,} splits, alpha={aa['expected_fpr']:.2f}")
print(f"Rejections:           {aa['rejections']:,}/{aa['n_sims']:,}")
print(f"FPR:                  {fpr:.4f}")
print(f"95% binomial CI:      [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"alpha=5% within CI:   {in_ci}")


Both arms heuristic, 10,000 splits, alpha=0.05
Rejections:           532/10,000
FPR:                  0.0532
95% binomial CI:      [0.0489, 0.0578]
alpha=5% within CI:   True


### D.5 Peeking (14 daily looks, stop-at-first-significance)

In [10]:
pk = ab['peeking']
print(f"Simulations: {pk['n_sims']}  Days: {pk['n_days']}  alpha={pk['alpha']}")
print(f"Fixed-horizon FPR:  {pk['fixed_horizon_false_positive_rate']:.3f}")
print(f"Peeking FPR:        {pk['peeking_false_positive_rate']:.3f}")
print(f"Inflation:          {pk['inflation_factor']:.2f}×")
print()
print("Peeking inflates the false-positive rate relative to waiting for the")
print("pre-specified sample size before evaluating.")


Simulations: 1000  Days: 14  alpha=0.05
Fixed-horizon FPR:  0.052
Peeking FPR:        0.224
Inflation:          4.31×

Peeking inflates the false-positive rate relative to waiting for the
pre-specified sample size before evaluating.


### D.6 Empirical vs Analytical Power

For each per-arm n, empirical power is the fraction of 1,000 simulation runs that rejected H0. Analytical power uses the inverse of the required-n formula: z_β = (δ·√n − z_{α/2}·√(2·p̄·q̄)) / √(p_A·q_A + p_B·q_B), power = Φ(z_β).

In [11]:
print(f"{'n/arm':>10s}  {'Empirical':>10s}  {'Analytical':>11s}  {'|diff|':>8s}")
print("-" * 46)
for pt in ab['empirical_power_curve']:
    ep  = pt.get('empirical_power', float('nan'))
    anl = pt.get('analytical_power', float('nan'))
    diff= pt.get('abs_diff', float('nan'))
    note = pt.get('note', '')
    ep_s  = f"{ep:.3f}"  if ep  == ep  else "n/a"
    anl_s = f"{anl:.3f}" if anl == anl else "n/a"
    d_s   = f"{diff:.3f}" if diff == diff else "n/a"
    print(f"  {pt['n_per_arm']:>8,d}  {ep_s:>10s}  {anl_s:>11s}  {d_s:>8s}  {note}")


     n/arm   Empirical   Analytical    |diff|
----------------------------------------------
       500       0.876        0.858     0.018  
     1,000       0.993        0.990     0.003  
     2,000       1.000        1.000     0.000  
     5,000       1.000        1.000     0.000  
    10,000       1.000        1.000     0.000  
    20,000       1.000        1.000     0.000  
    36,000       1.000        1.000     0.000  


### D.7 CUPED (primary metric: hit rate@12)

Two covariates compared on hit rate@12 variance reduction. Covariate (a): 4-week purchase count (pre-cutoff). Covariate (b): heuristic hit@12 in week 102 (pre-period same metric); customers not in week-102 GT set to 0 (tracked as n_no_pre_period_data). The better covariate (lower adj_factor) is used in the power table. AP@12 CUPED reported separately as secondary.

In [12]:
cu = ab['cuped']
ca = cu['covariate_a']
cb = cu['covariate_b']
best = cu['best_covariate']
print(f"Primary metric: {cu['primary_metric']}")
print()
print(f"{'Covariate':<35s}  {'corr':>6s}  {'var_red%':>9s}  {'n_req':>8s}")
print("-" * 65)
print(f"  (a) {ca['name']:<31s}  {ca['corr_with_hit_rate']:>6.4f}  "
      f"{ca['variance_reduction_pct']:>8.2f}%  {ca['n_required']:>8,d}")
n_no = cb.get('n_no_pre_period_data', 0)
n_with = cb.get('n_with_pre_period_data', '?')
print(f"  (b) {cb['name']:<31s}  {cb['corr_with_hit_rate']:>6.4f}  "
      f"{cb['variance_reduction_pct']:>8.2f}%  {cb['n_required']:>8,d}")
print(f"      ({n_with:,} with week-102 data, {n_no:,} set to 0)")
print()
print(f"Best covariate: ({best}) {cu['best_covariate_name']}")
print(f"n/arm without CUPED: {cu['n_required_unadjusted']:,}")
print(f"n/arm with CUPED:    {cu['n_required_cuped']:,}")
print()
sec = cu.get('secondary_ap12_covariate_a', {})
if sec:
    print(f"Secondary (AP@12, covariate a): corr={sec.get('covariate_correlation_with_metric',0):.4f}  "
          f"var_red={sec.get('variance_reduction_pct',0):.2f}%  "
          f"n_req={sec.get('n_required_cuped', sec.get('n_required', '?'))}")


Primary metric: hit_rate@12

Covariate                              corr   var_red%     n_req
-----------------------------------------------------------------
  (a) c_n_purchases_4w                 0.1619      2.62%       416
  (b) heuristic_hit12_week102          0.0992      0.98%       423
      (13,958 with week-102 data, 58,061 set to 0)

Best covariate: (a) c_n_purchases_4w
n/arm without CUPED: 427
n/arm with CUPED:    416

Secondary (AP@12, covariate a): corr=0.0858  var_red=0.74%  n_req=424


### D.8 Realistic Power Table

Required customers per arm and weeks (50/50 split, observed weekly buyers) for relative hit-rate lifts of 1%, 2%, 5%, 10%, and the observed lift; with and without CUPED. CUPED uses the better of the two covariates (labelled in the table).

In [13]:
cu = ab['cuped']
best_id = cu.get('best_covariate', '?')
best_nm = cu.get('best_covariate_name', '?')
pt_rows = ab.get('realistic_power_table', [])
print(f"CUPED covariate: ({best_id}) {best_nm}")
print(f"{'Lift':>10s}  {'p_treat':>8s}  {'n/arm (base)':>14s}  {'weeks':>8s}  "
      f"{'n/arm (CUPED)':>15s}  {'weeks':>8s}")
print("-" * 75)
for row in pt_rows:
    n_b = row.get('n_per_arm_no_cuped')
    n_c = row.get('n_per_arm_cuped')
    w_b = row.get('weeks_no_cuped')
    w_c = row.get('weeks_cuped')
    pt_p = row.get('p_treat')
    print(f"  {row['relative_lift']:>8s}  {(pt_p if pt_p else 0):>8.4f}  "
          f"{str(n_b) if n_b else '—':>14s}  {f'{w_b:.2f}' if w_b else '—':>8s}  "
          f"{str(n_c) if n_c else '—':>15s}  {f'{w_c:.2f}' if w_c else '—':>8s}")
pa = ab['power_analysis']
print(f"\nWeekly buying customers (fold 103): {pa['weekly_buying_customers_fold103']:,}")
print(f"Population caveat: {pa.get('population_caveat', '(see ab_test.json)')}")


CUPED covariate: (a) c_n_purchases_4w
      Lift   p_treat    n/arm (base)     weeks    n/arm (CUPED)     weeks
---------------------------------------------------------------------------
        1%    0.0866         1682326     46.72          1638250     45.49
        2%    0.0874          422475     11.73           411406     11.42
        5%    0.0900           68503      1.90            66708      1.85
       10%    0.0943           17502      0.49            17043      0.47
  observed    0.1471             427      0.01              416      0.01

Weekly buying customers (fold 103): 72,019
Population caveat: The offline metric is computed over customers who purchased in the target week. An online test would randomize all active visitors, lowering the baseline hit rate and increasing the required sample size.


## E — One-Shot Holdout (week 104)

### E.0 Protocol and Deviations

In [14]:
print("Protocol timestamp:", protocol['protocol_timestamp_utc'])
print("Labels-read timestamp:", holdout['labels_read_timestamp_utc'])
print(f"n eval customers: {holdout['n_eval_customers']:,}")
print()
devs = holdout.get('deviations', {})
print("Documented deviations:")
for key, val in devs.items():
    desc = val.get('description', str(val))
    print(f"  {key}: {desc[:120]}...")


Protocol timestamp: 2026-10-08T20:53:25.076540Z
Labels-read timestamp: 2026-10-08T21:23:46.171529+00:00
n eval customers: 68,984

Documented deviations:
  (a)_invalid_first_read: labels_104.parquet was first read in an earlier run at approximately 2026-10-08T21:11:35 UTC (recorded in PROJECT_LOG.md...
  (b)_fold103_downsampling: Fold-103 training negatives for the primary model (folds 100-103) were downsampled using .sample(fraction=0.2, seed=42) ...
  (c)_git_commit: The protocol's git_commit_hash (13ec5dc) identifies the last committed state at the time of the holdout run. Uncommitted...
  (d)_protocol_modification: holdout_protocol.json was modified after the label-read timestamp (mtime ≈ 2026-10-08T21:27 UTC, which is after labels_r...


**Deviations from ideal protocol (all documented in `holdout_results.json["deviations"]`):**

- **(a) Invalid first read:** labels_104.parquet was read in an earlier run (~2026-10-08T21:11:35Z) where ground truth was built without filtering label==1, producing Baseline B MAP@12=1.0. That run was discarded; ground truth was switched to `build_holdout_fold()`.
- **(b) Fold-103 downsampling:** Primary model's fold-103 negatives used `.sample(fraction=0.2, seed=42)` (random), not the hash-based rule used to build fold_103.parquet from scratch. Secondary model is unaffected. Primary MAP@12 = 0.036904, secondary = 0.036788; difference (+0.000116) is within the bootstrap CI.
- **(c) Git commit:** Protocol hash `13ec5dc` does not include uncommitted changes present at run time (scripts, evaluation modules).
- **(d) Protocol modification:** `run_holdout.py` originally added `labels_read_timestamp_utc` and `labels_sha256` to `holdout_protocol.json` after the label read. No pre-registered field changed; protocol has been restored to its pre-read state. Future runs use a version that writes only to `holdout_results.json`.


### E.1 Holdout Results

In [15]:
print(f"{'Model':<22s}  {'MAP@12':>9s}  {'95% CI':>26s}  {'NDCG@12':>9s}  {'Hit@12':>8s}")
print("-" * 85)
for name, key in [
    ("Primary (100-103)", "primary"),
    ("Secondary (100-102)", "secondary"),
    ("Heuristic", "heuristic"),
    ("Baseline B", "baseline_b"),
]:
    m = holdout['models'][key]
    ci = f"[{m['map@12_ci_lo']:.6f}, {m['map@12_ci_hi']:.6f}]"
    print(f"  {name:<20s}  {m['map@12']:>9.6f}  {ci:>26s}  "
          f"{m['ndcg@12']:>9.6f}  {m['hit_rate@12']:>8.4f}")

print()
pri = holdout['models']['primary']
bb_check = holdout['models']['baseline_b']
bvh = pri['boot_vs_heuristic']
bvb = pri['boot_vs_baseline_b']
print(f"Primary vs Heuristic: {bvh['mean_diff']:+.6f} CI=[{bvh['ci_lo']:+.6f}, {bvh['ci_hi']:+.6f}]")
print(f"Primary vs Baseline B: {bvb['mean_diff']:+.6f} CI=[{bvb['ci_lo']:+.6f}, {bvb['ci_hi']:+.6f}]")
print()
print(f"Baseline B MAP@12: {bb_check['map@12']:.6f}  (EDA-expected: {bb_check['eda_expected_map12']:.6f}  match: {bb_check['eda_match']})")
print(f"Primary lift vs heuristic: {holdout['primary_lift_vs_heuristic_pct']:.1f}%")
print(f"Primary lift vs Baseline B: {holdout['primary_lift_vs_baseline_b_pct']:.1f}%")


Model                      MAP@12                      95% CI    NDCG@12    Hit@12
-------------------------------------------------------------------------------------
  Primary (100-103)      0.036904        [0.035867, 0.037963]   0.054629    0.1522
  Secondary (100-102)    0.036788        [0.035811, 0.037906]   0.054254    0.1503
  Heuristic              0.024999        [0.024154, 0.025978]   0.034732    0.0859
  Baseline B             0.024457        [0.023626, 0.025447]   0.033912    0.0843

Primary vs Heuristic: +0.011904 CI=[+0.011273, +0.012582]
Primary vs Baseline B: +0.012447 CI=[+0.011778, +0.013105]

Baseline B MAP@12: 0.024457  (EDA-expected: 0.024457  match: True)
Primary lift vs heuristic: 47.6%
Primary lift vs Baseline B: 50.9%


**Holdout interpretation.**
The primary model (retrained on folds 100–103 with frozen hyperparameters) is evaluated once on week 104.
The protocol was pre-registered before any week-104 labels were read.
The Baseline B result matches the EDA-computed value of 0.024457 (ground truth for the recency-ordered heuristic).


## F — Limitations

1. **Offline replay assumption.** All results assume customers purchase the same items regardless of the list shown. Online feedback effects (exposure bias, position bias, novelty effects) are not captured.

2. **Temporal scope.** Lift is measured across folds 101–104 (weeks 101-104); the one-shot holdout itself covers a single week (week 104). Generalisation to other seasons or demand patterns is not measured.

3. **Candidate recall ceiling.** Recall@200 ≈ 17.5%; the ranker can only rank items that appear in the candidate pool. 82.5% of true purchases are outside the pool and cannot be recovered by a better ranker.

4. **Cold-start.** The 8.1% of target-week buyers with no prior history receive only popularity-based candidates; ranker features for these customers are sparse.

5. **A/B test is simulated.** The peeking and A/A analyses are computed on the same fold-103 data; a real online experiment is needed to measure conversion, engagement, and business KPIs.

6. **CUPED covariate.** Two covariates were evaluated on hit rate@12 variance reduction: the 4-week purchase count (`c_n_purchases_4w`, covariate a) and the pre-period heuristic hit@12 in week 102 (covariate b). Best covariate is `c_n_purchases_4w` with 2.6% variance reduction on hit rate@12 (corr=0.162, r²=0.026). At the observed lift, CUPED saves 11 customers per arm (427 → 416); at a 1% relative lift it saves 44,076 per arm (1,682,326 → 1,638,250). See section D.7 for the full covariate comparison.

7. **Online population caveat.** The offline metric is computed over customers who purchased in the target week. An online A/B test would randomise all active visitors, yielding a lower baseline hit rate and higher required sample size than the offline power analysis indicates.

## G — What Session 6 Builds On

- SHAP explainability: which features drive ranker predictions, and how they differ across segments.
- Serving export: ONNX or LightGBM native model file + `RecommenderState` serialisation for latency benchmarking.
- Online A/B test design: finalise the sample-size calculation and instrument the required logging.


## Number Sanity Check

In [16]:
# Key facts from JSON — verify they match the text above
assert abs(suite['ranker']['accuracy']['map@12'] - 0.035832) < 1e-4, "ranker MAP@12 mismatch"
assert abs(suite['heuristic']['accuracy']['hit_rate@12'] - 0.0857) < 0.001
assert abs(suite['ranker']['accuracy']['hit_rate@12'] - 0.1471) < 0.001
assert segs['n_flagged'] == 0
assert segs['n_segments_tested'] == 21
bb_map = holdout['models']['baseline_b']['map@12']
assert abs(bb_map - 0.024457) < 1e-4, f"Baseline B mismatch: {bb_map:.6f}"
# All segment p_adj must be > 0
for r in segs['results']:
    assert r.get('p_adj', 0) > 0, f"p_adj=0 in {r['family']}/{r['segment']}"
# T-test p-values must be present and in [0, 1]
for r in segs['results']:
    p_tt = r.get('p_ttest_adj')
    assert p_tt is not None, f"p_ttest_adj missing in {r['family']}/{r['segment']}"
    assert 0.0 <= p_tt <= 1.0, f"p_ttest_adj={p_tt} out of range"
# Holdout deviations must be present
assert 'deviations' in holdout, "holdout_results.json missing deviations section"
# AB test results present
assert 'aa_test' in ab, "ab_test.json missing aa_test"
assert 'peeking' in ab, "ab_test.json missing peeking"
assert 'cuped' in ab, "ab_test.json missing cuped"
assert 'realistic_power_table' in ab, "ab_test.json missing realistic_power_table"
# CUPED primary metric check
cu = ab['cuped']
assert cu.get('primary_metric') == 'hit_rate@12', "CUPED primary metric not hit_rate@12"
assert 'covariate_a' in cu and 'covariate_b' in cu, "CUPED missing covariate comparison"
assert cu.get('best_covariate') in ('a', 'b'), "CUPED best_covariate not set"
# A/A test: CI fields present, 10k sims
aa = ab['aa_test']
assert aa['n_sims'] >= 10000, f"A/A n_sims={aa['n_sims']}, expected >=10000"
assert 'fpr_ci_lo' in aa and 'fpr_ci_hi' in aa, "A/A missing binomial CI fields"
# Empirical power: analytical_power present for each finite entry
for pt in ab['empirical_power_curve']:
    if pt.get('empirical_power') is not None and pt.get('empirical_power') == pt.get('empirical_power'):
        assert 'analytical_power' in pt, f"analytical_power missing for n={pt['n_per_arm']}"
        assert 'abs_diff' in pt, f"abs_diff missing for n={pt['n_per_arm']}"
print(f"All sanity checks passed — 0 mismatches")


All sanity checks passed — 0 mismatches
